In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "modules"))

In [2]:
import time
import math
import numpy as np
from pyscf import gto


from misc_utils.pyscf_tools import get_integrals_rhf
from misc_utils.pyscf_tools import davidson_restricted_fullci_mask as davidson_pyscf
from misc_utils.matrix_utilities import davidson
from ormas_tools.ormas2 import ORMAS2

# Maximum CI dimension for memory safety                                                                                         
max_dim_ormas     =  200000000 # 0.2 billion                                                                                     
max_dim_fci_pyscf = 1000000000 # 1 billion

In [14]:
# We consider an Ar atom, with (9,9) electrons in total,                                         
# with 6-31G** basis, which amounts to 18 basis functions.                                       
R_H = 1.6
mol = gto.M(
    atom=f'''
    H {-3*R_H} 0 0
    H {-2*R_H} 0 0
    H   {-R_H} 0 0
    H    {R_H} 0 0
    H  {2*R_H} 0 0
    H  {3*R_H} 0 0
    ''',
    basis='6-31G',
    symmetry=False,
    verbose=0
)
# Atomic orbitals are characterized by                                                           
# 1s, 2s, 2p, 3s, 3p, 4s, 4p, and 3d orbitals in                                                 
# acsending order of orbital energy:                                                             
#  mo_energy =                                                                                   
#[-118.59460559  -12.31810372   -9.5684429    -9.5684429    -9.5684429                           
#   -1.27474354   -0.58891803   -0.58891803   -0.58891803    0.62666152                          
#    0.72602205    0.72602205    0.72602205    1.21762909    1.21762909                          
#    1.21762909    1.21762909    1.21762909]                                                     

In [18]:
n_fun = mol.nao        # total number of orbital functions
n_core = 0             # number of core orbitals
n_act = n_fun - n_core # number of active orbitals
n_elec = mol.nelec     # active electrons, each spin                                                  
n_elec_total = sum(n_elec) # active electrons

print(n_fun, n_core, n_act, n_elec, n_elec_total)

12 0 12 (3, 3) 6


In [6]:
# Restricted Active Space (RAS) example:                                                         
# RAS1 (2p, maximam hole = 2)                                                                    
# RAS2 (3s, 3p, 4s, 4p)                                                                          
# RAS3 (3d, maximam particle = 2)                                                                
#occ_info_spin = [
#    [
#        {'n_orb':6, 'min': -1, 'max': -1},
#    ],
#    [
#        {'n_orb':6, 'min': -1, 'max': -1},
#    ]
#]
occ_info_spin = [
    [
        {'n_orb':6, 'min':-1, 'max':-1},
        {'n_orb':6, 'min': 0, 'max': 2},
    ],
    [
        {'n_orb':6, 'min':-1, 'max':-1},
        {'n_orb':6, 'min': 0, 'max': 2},
    ]
]
#occ_info_total = [
#        {'n_orb':12, 'min':-1, 'max': -1},
#]
occ_info_total = [
        {'n_orb':12, 'min':-1, 'max':-1},
        {'n_orb':12, 'min': 0, 'max': 2},
]

In [7]:
########################################################################                         
# CI instance generation                                                                         
print("\n")
print("#"*72)
print("[Step 1] Constructing CI object...")
t0 = time.time()
myCI = ORMAS2(n_elec,
              occ_info_spin,
              occ_info_total,
              num_threads = 10,
              verbose = 1)
print(f" ... Done: {time.time()-t0:.2f} sec.")
print(f"CI dimension: {myCI.total_dim}")
print("String distribution over occupation groups:")
print(myCI.mat_num_str)



########################################################################
[Step 1] Constructing CI object...
ORMAS1: num_threads =  10
ORMAS1: Allowed distributions 0.000 seconds
ORMAS1: generate strings 0.505 seconds
ORMAS1: build_string_to_index 0.337 seconds
ORMAS1: _generate_trans1 1.725 seconds
ORMAS1: _generate_trans2 0.617 seconds
ORMAS1: num_threads =  10
ORMAS1: Allowed distributions 0.000 seconds
ORMAS1: generate strings 0.000 seconds
ORMAS1: build_string_to_index 0.000 seconds
ORMAS1: _generate_trans1 0.001 seconds
ORMAS1: _generate_trans2 0.002 seconds
ORMAS1 objects construction 3.995 seconds
generate_distributions 0.000 seconds
generate_occupation_boundaries 0.000 seconds
generate_determinant_offsets 0.008 seconds
generate_transpose_map 0.002 seconds
generate_trans1_det_boundaries 0.123 seconds
generate_trans2_det_boundaries 0.002 seconds
generate_trans1_for_pq_valid 0.350 seconds
 ... Done: 4.48 sec.
CI dimension: 15700
String distribution over occupation groups:
[[ 400

In [11]:
########################################################################                         
# Hamiltonian matrix elements generation                                                         
print("\n")
print("#"*72)
print("[Step 2] Running RHF to obtain MO integrals...")
t0 = time.time()
e_core, h1eff, h2_phys = get_integrals_rhf(mol, n_core, n_act, n_elec_total)
print(f" ... Done: {time.time()-t0:.2f} sec.")
print(f"Core energy: {e_core}")



########################################################################
[Step 2] Running RHF to obtain MO integrals...
RHF Total Energy: -2.8128608644 Hartree (Time: 0.03 sec)
Integral transformation completed (Time: 0.00 sec)
 ... Done: 0.03 sec.
Core energy: 2.475005913573749


In [12]:
########################################################################                         
# Direct-CI Davidson diagonalization
print("\n")
print("#"*72)
if myCI.total_dim < max_dim_ormas:
    print("[Step 3] ORMAS Davidson diagonalization...")
    t0 = time.time()
    def my_get_sigma(x):
        return myCI.h_prod(h1eff, h2_phys, x).real
        #return myCI.h_prod_force_symmetric(h1eff, h2_phys, x).real                              
        return get_sigma(x).real

    hdiag = myCI.calc_hdiag(h1eff, h2_phys).real
    def my_precond(dx, e, x0):
        denom = hdiag - e
        denom[abs(denom) < 1e-8] = 1e-8  # avoid zero division                                   
        return dx / denom

    E1, U1 = davidson(myCI.total_dim,
                      my_get_sigma,
                      my_precond,
                      verbose=5)
    print(f" ... Done: {time.time()-t0:.2f} sec.")
    print(f"ORMAS CI energy: {E1 + e_core}")
else:
    print("[Step 3] myCI.total_dim too large. Skip ORMAS diagonalization.")




########################################################################
[Step 3] ORMAS Davidson diagonalization...
verbose = 5
davidson 0 1  |r|= 0.359  e= [-5.28786678]  max|de|= -5.29  lindep=    1
davidson 1 2  |r|= 0.302  e= [-5.43791731]  max|de|= -0.15  lindep= 0.979
davidson 2 3  |r|= 0.18  e= [-5.54185702]  max|de|= -0.104  lindep= 0.989
davidson 3 4  |r|= 0.103  e= [-5.568876]  max|de|= -0.027  lindep= 0.954
davidson 4 5  |r|= 0.0458  e= [-5.57595083]  max|de|= -0.00707  lindep= 0.956
davidson 5 6  |r|= 0.0188  e= [-5.57706126]  max|de|= -0.00111  lindep= 0.978
davidson 6 7  |r|= 0.00998  e= [-5.57723465]  max|de|= -0.000173  lindep= 0.982
davidson 7 8  |r|= 0.00771  e= [-5.57730833]  max|de|= -7.37e-05  lindep= 0.942
davidson 8 9  |r|= 0.00898  e= [-5.57738101]  max|de|= -7.27e-05  lindep= 0.904
davidson 9 10  |r|= 0.00587  e= [-5.57745741]  max|de|= -7.64e-05  lindep= 0.943
davidson 10 11  |r|= 0.00196  e= [-5.57747476]  max|de|= -1.74e-05  lindep= 0.962
davidson 11 12  |